# R3D-18 3D CNN Study

Systematic comparison of R3D-18 configurations for per-rep push-up form classification.

**Data source:** Manually annotated reps with human-labeled frame boundaries (no state machine).

**Study design:**
1. **Experiment A**: Full-frame input (frozen backbone)
2. **Experiment B**: YOLO-crop input (frozen backbone)
3. **Winner**: Unfreeze backbone layers (1 block, 2 blocks)
4. **Hyperparameter tuning**: LR + batch size sweep on best config
5. **Final comparison** table + plots

All experiments use 5-fold stratified CV split by video to prevent data leakage.

## Setup + Data Loading

In [1]:
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from model import PushUpR3D
from datasets import PushUpRepVideoDataset, PushUpRepCroppedVideoDataset
from training import run_rep_kfold_cv
from data_loader import load_annotations, extract_keypoints, attach_keypoints

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================================================
# CONFIGURATION — adjust these paths for your environment
# ============================================================
ONEDRIVE_DIR = Path.home() / "Library/CloudStorage/OneDrive-SingaporeManagementUniversity"
DATASET_DIR  = ONEDRIVE_DIR / "Group 4 Deep Learning for Computer Vision Project - training_dataset"

ANNOTATIONS   = DATASET_DIR / "annotations_template.xlsx"  # CSV or XLSX
VIDEO_DIR     = DATASET_DIR / "videos"                      # all .mp4 files here
KEYPOINT_DIR  = Path("keypoints")                           # YOLO keypoints saved locally
OUTPUT_DIR    = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CV settings
RANDOM_STATE = 42
N_SPLITS     = 5
N_FRAMES     = 16   # frames sampled per rep

# Device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")
print(f"Video dir: {VIDEO_DIR}")
print(f"Annotations: {ANNOTATIONS}")

Device: mps
Video dir: /Users/ikhyvicky/Library/CloudStorage/OneDrive-SingaporeManagementUniversity/Group 4 Deep Learning for Computer Vision Project - training_dataset/videos
Annotations: /Users/ikhyvicky/Library/CloudStorage/OneDrive-SingaporeManagementUniversity/Group 4 Deep Learning for Computer Vision Project - training_dataset/annotations_template.xlsx


In [2]:
# Load manually annotated rep boundaries (supports CSV and XLSX)
rep_segments = load_annotations(ANNOTATIONS, VIDEO_DIR)

n_good = sum(1 for r in rep_segments if r["label"] == 0)
n_bad  = sum(1 for r in rep_segments if r["label"] == 1)
unique_videos = set(r["video_id"] for r in rep_segments)

print(f"Total reps: {len(rep_segments)} (good={n_good}, bad={n_bad})")
print(f"Unique videos: {len(unique_videos)}")
print(f"Mean reps/video: {len(rep_segments)/max(len(unique_videos),1):.1f}")

# Rep length distribution
if rep_segments:
    rep_lengths = [r["end_frame"] - r["start_frame"] for r in rep_segments]
    print(f"Rep length (frames): min={min(rep_lengths)}, max={max(rep_lengths)}, "
          f"median={np.median(rep_lengths):.0f}")

INFO: Loaded 272 reps from annotations_template.xlsx (skipped 6)


Total reps: 272 (good=126, bad=146)
Unique videos: 203
Mean reps/video: 1.3
Rep length (frames): min=16, max=396, median=84


## Extract YOLO Keypoints (one-time)

Runs YOLO pose estimation on all videos and saves keypoints as `.npy` files.
Skips videos that already have keypoints extracted. Only needed for Experiment B (YOLO-crop).

In [3]:
# Extract keypoints (skips already-extracted videos)
extract_keypoints(rep_segments, KEYPOINT_DIR)

# Attach keypoints to rep dicts for the cropped dataset
attach_keypoints(rep_segments, KEYPOINT_DIR)

# Verify
has_kps = sum(1 for r in rep_segments if "keypoints" in r)
print(f"Reps with keypoints: {has_kps}/{len(rep_segments)}")

INFO: Extracting keypoints: 1
INFO:   Saved: 1.npy (290 frames)
INFO: Extracting keypoints: 10
INFO:   Saved: 10.npy (126 frames)
INFO: Extracting keypoints: 11
INFO:   Saved: 11.npy (83 frames)
INFO: Extracting keypoints: 12
INFO:   Saved: 12.npy (121 frames)
INFO: Extracting keypoints: 13
INFO:   Saved: 13.npy (121 frames)
INFO: Extracting keypoints: 14
INFO:   Saved: 14.npy (125 frames)
INFO: Extracting keypoints: 15
INFO:   Saved: 15.npy (100 frames)
INFO: Extracting keypoints: 16
INFO:   Saved: 16.npy (100 frames)
INFO: Extracting keypoints: 17
INFO:   Saved: 17.npy (107 frames)
INFO: Extracting keypoints: 18
INFO:   Saved: 18.npy (153 frames)
INFO: Extracting keypoints: 19
INFO:   Saved: 19.npy (126 frames)
INFO: Extracting keypoints: 2
INFO:   Saved: 2.npy (158 frames)
INFO: Extracting keypoints: 20
INFO:   Saved: 20.npy (93 frames)
INFO: Extracting keypoints: 21
INFO:   Saved: 21.npy (105 frames)
INFO: Extracting keypoints: 22
INFO:   Saved: 22.npy (250 frames)
INFO: Extracting

Reps with keypoints: 272/272


---
## Experiment A: Full-Frame R3D-18 (Frozen Backbone)

Standard preprocessing: resize to 128x171, center crop to 112x112.
Only the FC head is trainable (1,026 params).

In [ ]:
def make_model_frozen():
    return PushUpR3D(freeze_backbone=True)

def make_full_frame_dataset(reps):
    return PushUpRepVideoDataset(reps, n_frames=N_FRAMES, augment=False)

def make_full_frame_dataset_aug(reps):
    return PushUpRepVideoDataset(reps, n_frames=N_FRAMES, augment=True)

print(f"Experiment A: Full-frame, frozen backbone")
print(f"  Trainable params: {make_model_frozen().trainable_param_count():,}")
print(f"  Augmentation: horizontal flip, random crop, temporal jitter, color jitter")

t0 = time.time()
results_A = run_rep_kfold_cv(
    model_factory=make_model_frozen,
    dataset_factory=make_full_frame_dataset,
    rep_segments=rep_segments,
    n_splits=N_SPLITS,
    n_epochs=30,
    batch_size=8,
    lr=1e-3,
    patience=10,
    device_str=DEVICE,
    random_state=RANDOM_STATE,
    train_dataset_factory=make_full_frame_dataset_aug,
)
time_A = time.time() - t0

accs_A = [f["val_accuracy"] for f in results_A["fold_results"]]
print(f"\nExperiment A results:")
print(f"  Per-fold: {[f'{a:.1%}' for a in accs_A]}")
print(f"  Mean: {np.mean(accs_A):.1%} +/- {np.std(accs_A):.1%}")
print(f"  Time: {time_A:.0f}s")

---
## Experiment B: YOLO-Crop R3D-18 (Frozen Backbone)

Uses YOLO keypoints to compute per-frame bounding box, crops to the person,
resizes crop to 112x112. Same frozen backbone, only FC trainable.

In [ ]:
# Filter to reps that have keypoints attached
reps_with_kps = [r for r in rep_segments if "keypoints" in r]
print(f"Reps with keypoints for YOLO-crop: {len(reps_with_kps)}/{len(rep_segments)}")

def make_cropped_dataset(reps):
    return PushUpRepCroppedVideoDataset(reps, n_frames=N_FRAMES, augment=False)

def make_cropped_dataset_aug(reps):
    return PushUpRepCroppedVideoDataset(reps, n_frames=N_FRAMES, augment=True)

print(f"\nExperiment B: YOLO-crop, frozen backbone")
print(f"  Trainable params: {make_model_frozen().trainable_param_count():,}")

t0 = time.time()
results_B = run_rep_kfold_cv(
    model_factory=make_model_frozen,
    dataset_factory=make_cropped_dataset,
    rep_segments=reps_with_kps,
    n_splits=N_SPLITS,
    n_epochs=30,
    batch_size=8,
    lr=1e-3,
    patience=10,
    device_str=DEVICE,
    random_state=RANDOM_STATE,
    train_dataset_factory=make_cropped_dataset_aug,
)
time_B = time.time() - t0

accs_B = [f["val_accuracy"] for f in results_B["fold_results"]]
print(f"\nExperiment B results:")
print(f"  Per-fold: {[f'{a:.1%}' for a in accs_B]}")
print(f"  Mean: {np.mean(accs_B):.1%} +/- {np.std(accs_B):.1%}")
print(f"  Time: {time_B:.0f}s")

---
## Compare A vs B

Pick the winner (higher mean accuracy) for the unfreezing experiments.

In [ ]:
mean_A = np.mean(accs_A)
mean_B = np.mean(accs_B)

comparison_ab = pd.DataFrame({
    "Experiment": ["A: Full-frame", "B: YOLO-crop"],
    "Mean Accuracy": [f"{mean_A:.1%}", f"{mean_B:.1%}"],
    "Std": [f"{np.std(accs_A):.1%}", f"{np.std(accs_B):.1%}"],
    "Per-Fold": [str([f'{a:.1%}' for a in accs_A]), str([f'{a:.1%}' for a in accs_B])],
    "Time (s)": [f"{time_A:.0f}", f"{time_B:.0f}"],
})
print(comparison_ab.to_string(index=False))

if mean_B > mean_A:
    WINNER_INPUT = "crop"
    winner_factory = make_cropped_dataset
    winner_factory_aug = make_cropped_dataset_aug
    winner_reps = reps_with_kps
    winner_name = "B: YOLO-crop"
    winner_accs = accs_B
    winner_results = results_B
    winner_time = time_B
else:
    WINNER_INPUT = "full"
    winner_factory = make_full_frame_dataset
    winner_factory_aug = make_full_frame_dataset_aug
    winner_reps = rep_segments
    winner_name = "A: Full-frame"
    winner_accs = accs_A
    winner_results = results_A
    winner_time = time_A

print(f"\nWinner: {winner_name} ({np.mean(winner_accs):.1%})")
print(f"Proceeding with {WINNER_INPUT} input for unfreezing experiments.")

# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["A: Full-frame", "B: YOLO-crop"],
    [mean_A, mean_B],
    yerr=[np.std(accs_A), np.std(accs_B)],
    capsize=8, color=["#4C72B0", "#DD8452"], alpha=0.85, edgecolor="black",
)
for bar, m in zip(bars, [mean_A, mean_B]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{m:.1%}", ha="center", fontweight="bold")
ax.set_ylabel("Accuracy")
ax.set_title("Experiment A vs B: Input Preprocessing")
ax.set_ylim(0, 1.15)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Chance")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ab_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Unfreezing Experiment

Using the winning input preprocessing, compare:
- **Frozen**: Only FC head trainable (baseline — already done above)
- **Unfreeze 1 block**: layer4 + FC trainable
- **Unfreeze 2 blocks**: layer3 + layer4 + FC trainable

In [ ]:
unfreeze_results = {}

# Already have frozen results from the winner
unfreeze_results["frozen"] = {
    "accs": winner_accs,
    "results": winner_results,
    "time": winner_time,
    "params": make_model_frozen().trainable_param_count(),
}

for n_blocks in [1, 2]:
    def make_model_unfrozen(n=n_blocks):
        m = PushUpR3D(freeze_backbone=True)
        m.unfreeze_last_n_blocks(n)
        return m

    n_params = make_model_unfrozen().trainable_param_count()
    label = f"unfreeze_{n_blocks}"
    print(f"\n{'='*60}")
    print(f"Unfreeze {n_blocks} block(s): {n_params:,} trainable params")
    print(f"{'='*60}")

    t0 = time.time()
    res = run_rep_kfold_cv(
        model_factory=make_model_unfrozen,
        dataset_factory=winner_factory,
        rep_segments=winner_reps,
        n_splits=N_SPLITS,
        n_epochs=30,
        batch_size=8,
        lr=1e-4,    # lower LR for fine-tuning unfrozen layers
        patience=10,
        device_str=DEVICE,
        random_state=RANDOM_STATE,
        train_dataset_factory=winner_factory_aug,
    )
    elapsed = time.time() - t0

    accs = [f["val_accuracy"] for f in res["fold_results"]]
    print(f"  Per-fold: {[f'{a:.1%}' for a in accs]}")
    print(f"  Mean: {np.mean(accs):.1%} +/- {np.std(accs):.1%}")
    print(f"  Time: {elapsed:.0f}s")

    unfreeze_results[label] = {
        "accs": accs,
        "results": res,
        "time": elapsed,
        "params": n_params,
    }

In [ ]:
# Unfreezing comparison table
unfreeze_labels = ["frozen", "unfreeze_1", "unfreeze_2"]
display_names = ["Frozen (FC only)", "Unfreeze 1 block (layer4)", "Unfreeze 2 blocks (layer3+4)"]

unfreeze_df = pd.DataFrame({
    "Config": display_names,
    "Trainable Params": [f"{unfreeze_results[k]['params']:,}" for k in unfreeze_labels],
    "Mean Accuracy": [f"{np.mean(unfreeze_results[k]['accs']):.1%}" for k in unfreeze_labels],
    "Std": [f"{np.std(unfreeze_results[k]['accs']):.1%}" for k in unfreeze_labels],
    "Time (s)": [f"{unfreeze_results[k]['time']:.0f}" for k in unfreeze_labels],
})
print(unfreeze_df.to_string(index=False))

# Bar chart
means_u = [np.mean(unfreeze_results[k]["accs"]) for k in unfreeze_labels]
stds_u = [np.std(unfreeze_results[k]["accs"]) for k in unfreeze_labels]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#4C72B0", "#55A868", "#C44E52"]
bars = ax.bar(display_names, means_u, yerr=stds_u, capsize=8,
              color=colors, alpha=0.85, edgecolor="black")
for bar, m, s in zip(bars, means_u, stds_u):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.01,
            f"{m:.1%}", ha="center", fontweight="bold")
ax.set_ylabel("Accuracy")
ax.set_title(f"Unfreezing Study ({winner_name} input)")
ax.set_ylim(0, 1.15)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "unfreeze_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Pick best unfreezing config
best_unfreeze = max(unfreeze_labels, key=lambda k: np.mean(unfreeze_results[k]["accs"]))
best_unfreeze_acc = np.mean(unfreeze_results[best_unfreeze]["accs"])
print(f"\nBest unfreezing config: {best_unfreeze} ({best_unfreeze_acc:.1%})")

---
## Hyperparameter Tuning

Grid search over learning rate and batch size using the best config from above.

In [ ]:
# Determine the best model factory based on unfreezing results
def _make_best_model():
    m = PushUpR3D(freeze_backbone=True)
    if best_unfreeze == "unfreeze_1":
        m.unfreeze_last_n_blocks(1)
    elif best_unfreeze == "unfreeze_2":
        m.unfreeze_last_n_blocks(2)
    return m

print(f"Tuning config: {best_unfreeze} + {WINNER_INPUT} input")
print(f"  Trainable params: {_make_best_model().trainable_param_count():,}")

# Hyperparameter grid
LR_OPTIONS = [1e-4, 5e-4, 1e-3]
BS_OPTIONS = [4, 8, 16]

hp_results = []

for lr in LR_OPTIONS:
    for bs in BS_OPTIONS:
        print(f"\n--- lr={lr}, batch_size={bs} ---")
        t0 = time.time()
        res = run_rep_kfold_cv(
            model_factory=_make_best_model,
            dataset_factory=winner_factory,
            rep_segments=winner_reps,
            n_splits=N_SPLITS,
            n_epochs=30,
            batch_size=bs,
            lr=lr,
            patience=10,
            device_str=DEVICE,
            random_state=RANDOM_STATE,
            train_dataset_factory=winner_factory_aug,
        )
        elapsed = time.time() - t0
        accs = [f["val_accuracy"] for f in res["fold_results"]]
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)
        print(f"  Mean: {mean_acc:.1%} +/- {std_acc:.1%} ({elapsed:.0f}s)")

        hp_results.append({
            "lr": lr,
            "batch_size": bs,
            "mean_acc": mean_acc,
            "std_acc": std_acc,
            "per_fold": accs,
            "time": elapsed,
            "results": res,
        })

In [ ]:
# Hyperparameter results table
hp_df = pd.DataFrame([
    {
        "LR": f"{r['lr']:.0e}",
        "Batch Size": r["batch_size"],
        "Mean Accuracy": f"{r['mean_acc']:.1%}",
        "Std": f"{r['std_acc']:.1%}",
        "Time (s)": f"{r['time']:.0f}",
    }
    for r in hp_results
])
print(hp_df.to_string(index=False))

# Heatmap
pivot = pd.DataFrame(hp_results).pivot(
    index="batch_size", columns="lr", values="mean_acc"
)

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto", vmin=0.5, vmax=1.0)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{c:.0e}" for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("Learning Rate")
ax.set_ylabel("Batch Size")
ax.set_title("Hyperparameter Grid Search")

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f"{val:.1%}", ha="center", va="center",
                fontweight="bold", color="white" if val > 0.75 else "black")

plt.colorbar(im, label="Accuracy")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "hp_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# Best HP config
best_hp = max(hp_results, key=lambda r: r["mean_acc"])
print(f"\nBest hyperparams: lr={best_hp['lr']:.0e}, batch_size={best_hp['batch_size']}")
print(f"  Accuracy: {best_hp['mean_acc']:.1%} +/- {best_hp['std_acc']:.1%}")

---
## Final Comparison

Summary of all experiments in a single table and chart.

In [ ]:
# Build final comparison
all_experiments = []

# A: full-frame frozen
all_experiments.append({
    "Experiment": "A: Full-frame (frozen)",
    "Input": "Full-frame",
    "Backbone": "Frozen",
    "Params": f"{make_model_frozen().trainable_param_count():,}",
    "LR": "1e-3",
    "Batch Size": 8,
    "Mean Acc": np.mean(accs_A),
    "Std": np.std(accs_A),
    "Time (s)": f"{time_A:.0f}",
})

# B: YOLO-crop frozen
all_experiments.append({
    "Experiment": "B: YOLO-crop (frozen)",
    "Input": "YOLO-crop",
    "Backbone": "Frozen",
    "Params": f"{make_model_frozen().trainable_param_count():,}",
    "LR": "1e-3",
    "Batch Size": 8,
    "Mean Acc": np.mean(accs_B),
    "Std": np.std(accs_B),
    "Time (s)": f"{time_B:.0f}",
})

# Unfreezing experiments
for label, display in [("unfreeze_1", "Unfreeze 1 block"), ("unfreeze_2", "Unfreeze 2 blocks")]:
    r = unfreeze_results[label]
    all_experiments.append({
        "Experiment": f"{display} ({WINNER_INPUT})",
        "Input": WINNER_INPUT.capitalize(),
        "Backbone": display,
        "Params": f"{r['params']:,}",
        "LR": "1e-4",
        "Batch Size": 8,
        "Mean Acc": np.mean(r["accs"]),
        "Std": np.std(r["accs"]),
        "Time (s)": f"{r['time']:.0f}",
    })

# Best HP
all_experiments.append({
    "Experiment": f"Best HP ({best_unfreeze}, {WINNER_INPUT})",
    "Input": WINNER_INPUT.capitalize(),
    "Backbone": best_unfreeze.replace("_", " ").title(),
    "Params": f"{_make_best_model().trainable_param_count():,}",
    "LR": f"{best_hp['lr']:.0e}",
    "Batch Size": best_hp["batch_size"],
    "Mean Acc": best_hp["mean_acc"],
    "Std": best_hp["std_acc"],
    "Time (s)": f"{best_hp['time']:.0f}",
})

final_df = pd.DataFrame(all_experiments)
print(final_df.to_string(index=False))

In [ ]:
# Final bar chart
names = final_df["Experiment"].tolist()
means_f = final_df["Mean Acc"].tolist()
stds_f = final_df["Std"].tolist()

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(names)))
bars = ax.bar(range(len(names)), means_f, yerr=stds_f, capsize=6,
              color=colors, alpha=0.85, edgecolor="black")
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)

for bar, m, s in zip(bars, means_f, stds_f):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.01,
            f"{m:.1%}", ha="center", fontweight="bold", fontsize=9)

ax.set_ylabel("Accuracy")
ax.set_title(f"R3D-18 Study: All Configurations ({len(rep_segments)} reps, {N_SPLITS}-fold CV)")
ax.set_ylim(0, 1.15)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Highlight best
best_row = final_df.loc[final_df["Mean Acc"].idxmax()]
print(f"\nBest overall: {best_row['Experiment']}")
print(f"  Accuracy: {best_row['Mean Acc']:.1%} +/- {best_row['Std']:.1%}")
print(f"  Config: {best_row['Params']} params, lr={best_row['LR']}, bs={best_row['Batch Size']}")

---
## Save Results

In [ ]:
# Save summary CSV
final_df.to_csv(OUTPUT_DIR / "r3d_study_results.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'r3d_study_results.csv'}")

# Save HP grid CSV
hp_df.to_csv(OUTPUT_DIR / "r3d_hp_grid.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'r3d_hp_grid.csv'}")

# Save best model weights
if best_hp["results"]["best_state"] is not None:
    torch.save(best_hp["results"]["best_state"], OUTPUT_DIR / "r3d_best.pt")
    print(f"Saved: {OUTPUT_DIR / 'r3d_best.pt'}")

print(f"\nAll outputs saved to: {OUTPUT_DIR.resolve()}")
print("\nFigures:")
for f in sorted(OUTPUT_DIR.glob("*.png")):
    print(f"  {f.name}")
print("CSVs:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {f.name}")